In [1]:
import math

def norm_cdf(x):
    """
    计算标准正态分布的累积分布函数 N(x)
    使用 math.erf (误差函数) 实现，精度极高
    """
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def bs_call_price(S, K, T, r, q, sigma):
    """
    Black-Scholes 看涨期权定价公式
    """
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    
    price = S * math.exp(-q * T) * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return price

def bs_call_delta(S, K, T, r, q, sigma):
    """
    Black-Scholes 看涨期权 Delta 公式
    Delta = e^(-qT) * N(d1)
    """
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return math.exp(-q * T) * norm_cdf(d1)

def find_implied_vol(target_price, S, K, T, r, q):
    """
    使用二分查找法 (Binary Search) 反推隐含波动率
    精度设置为 1e-8
    """
    low = 0.0001
    high = 5.0  # 假设波动率不会超过 500%
    tolerance = 1e-8
    
    for i in range(100): # 最多迭代100次
        mid = (low + high) / 2
        price = bs_call_price(S, K, T, r, q, mid)
        
        if abs(price - target_price) < tolerance:
            return mid
        
        if price < target_price:
            low = mid
        else:
            high = mid
            
    return (low + high) / 2


In [2]:

# ==========================================
# 主程序逻辑
# ==========================================

# 1. 设定参数
S = 55.0        # 标的价格
r = 0.04        # 无风险利率
q = 0.03        # 股息率
T = 1.0         # 到期时间
f_market = 10.94 # 衍生品 f 的市场价格

# 2. 计算 K=50 看涨期权的理论价格 (已知 Vol = 35%)
K1 = 50
sigma_1 = 0.35
c_50_price = bs_call_price(S, K1, T, r, q, sigma_1)

print("-" * 40)
print(f"步骤 1: 计算 C(50) 价格 (Sigma={sigma_1:.0%})")
print(f"C(50) = {c_50_price:.6f}") 
# 预期结果应接近 10.0247

# 3. 剥离出 K=100 看涨期权的市场隐含价格
# Payoff = C(50) + C(100) -> f = Price(50) + Price(100)
c_100_target_price = f_market - c_50_price

print("-" * 40)
print(f"步骤 2: 推导 C(100) 的市场价格")
print(f"C(100) = f_market - C(50)")
print(f"C(100) = {10.94} - {c_50_price:.6f} = {c_100_target_price:.6f}")
# 预期结果应接近 0.9153

# 4. 反推 K=100 的隐含波动率
K2 = 100
sigma_2 = find_implied_vol(c_100_target_price, S, K2, T, r, q)

print("-" * 40)
print(f"步骤 3: 反推 C(100) 的隐含波动率")
print(f"Implied Vol (K=100) = {sigma_2:.6f} ({sigma_2:.4%})")
# 预期结果应接近 40.30%

# 5. 计算 Delta
# 注意：分别使用各自的波动率计算
delta_50 = bs_call_delta(S, K1, T, r, q, sigma_1)
delta_100 = bs_call_delta(S, K2, T, r, q, sigma_2)
total_delta = delta_50 + delta_100

print("-" * 40)
print(f"步骤 4: 计算 Delta (b问)")
print(f"Delta(C50)  [Sigma={sigma_1:.2%}] = {delta_50:.6f}")
print(f"Delta(C100) [Sigma={sigma_2:.2%}] = {delta_100:.6f}")
print(f"Total Delta = {total_delta:.6f}")
print("-" * 40)

# 最终验证结论
print("结论验证：")
print(f"同学的答案: 0.7640")
print(f"程序计算值: {total_delta:.4f}")

----------------------------------------
步骤 1: 计算 C(50) 价格 (Sigma=35%)
C(50) = 10.024653
----------------------------------------
步骤 2: 推导 C(100) 的市场价格
C(100) = f_market - C(50)
C(100) = 10.94 - 10.024653 = 0.915347
----------------------------------------
步骤 3: 反推 C(100) 的隐含波动率
Implied Vol (K=100) = 0.403001 (40.3001%)
----------------------------------------
步骤 4: 计算 Delta (b问)
Delta(C50)  [Sigma=35.00%] = 0.662739
Delta(C100) [Sigma=40.30%] = 0.101266
Total Delta = 0.764004
----------------------------------------
结论验证：
同学的答案: 0.7640
程序计算值: 0.7640
